# Dự đoán Bệnh Tiểu đường — Lập trình & Xử lý Dữ liệu (Medical AI)
**Dataset:** CDC Diabetes Health Indicators (BRFSS 2015) — UCI ID 891 — 253.680 mẫu, 21 đặc trưng + target

Mô tả bài toán: phân loại nhị phân (binary classification) có sự mất cân bằng nhãn . Từ các chỉ số sức khỏe và lối sống của một người, mô hình dự đoán người đó có bị tiểu đường hay không.

Dữ liệu: CDC BRFSS 2015, khảo sát sức khỏe ở Mỹ, gồm 253.680 người, 21 đặc trưng và 1 nhãn Diabetes_binary.

Nhãn: 0 là không tiểu đường, 1 là tiền tiểu đường hoặc tiểu đường.




## 1. CHUẨN BỊ MÔI TRƯỜNG

In [ ]:
import warnings
import textwrap


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

try:
    from imblearn.over_sampling import SMOTE, SMOTENC
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. TẢI DỮ LIỆU

In [ ]:
%pip install kagglehub imbalanced-learn
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
data_folder = Path(path)

csv_file = data_folder / "diabetes_binary_health_indicators_BRFSS2015.csv"
df = pd.read_csv(csv_file)

TARGET = "Diabetes_binary"
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

## 3. Khám phá & Tiền xử lý dữ liệu (EDA & Clean Data)
Mục tiêu: khảo sát phân phối, phát hiện missing values, ngoại lai, tương quan và chuẩn bị cho scaling.

In [ ]:
print("── Thông tin kiểu dữ liệu ──")
print(df.dtypes.to_string())
print("\n── Thống kê mô tả ──")
display(df.describe().T)

print("\n── Tỉ lệ target ──")
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).mul(100).round(2).astype(str) + " %")

miss = df.isna().sum()
miss = miss[miss>0].sort_values(ascending=False)
print("\n── Missing values ──")
print(miss.to_string() if len(miss)>0 else "Không có missing (hoặc đã được làm sạch sẵn)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Phân phối các biến liên tục / thứ bậc", fontsize=13, weight="bold")

for ax, col in zip(axes.flat, ["BMI","GenHlth","MentHlth","PhysHlth","Age","Income"]):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, bins=30, color="#4C78A8")
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(data=df, x=TARGET, y="BMI", palette=["#72B7B2","#E45756"], legend=False, ax=ax)
ax.set_title("BMI theo nhóm Diabetes (phát hiện ngoại lai)")
ax.set_xlabel("Diabetes (0=không, 1=có)")
plt.show()

In [ ]:
num_cols = ["BMI","GenHlth","MentHlth","PhysHlth","Age","Education","Income"]
corr = df[num_cols + [TARGET]].corr(numeric_only=True)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-0.4, vmax=0.6, square=True)
plt.title("Ma trận tương quan (Pearson) — các biến số & target", weight="bold")
plt.tight_layout()
plt.show()

print("Tương quan với Diabetes_binary:")
print(corr[TARGET].sort_values(ascending=False).to_string())

In [ ]:
bin_cols = ["HighBP","HighChol","Smoker","Stroke","HeartDiseaseorAttack",
            "PhysActivity","Fruits","Veggies","HvyAlcoholConsump","DiffWalk","Sex"]

fig, axes = plt.subplots(3, 4, figsize=(15, 9))
fig.suptitle("Tỉ lệ Diabetes theo các biến nhị phân", fontsize=13, weight="bold")
axes = axes.flat
for i, col in enumerate(bin_cols):
    tmp = df.groupby(col)[TARGET].mean()
    axes[i].bar(tmp.index.astype(str), tmp.values, color=["#72B7B2","#E45756"][:len(tmp)])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylim(0, 0.45)
    for j, v in enumerate(tmp.values):
        axes[i].text(j, v+0.01, f"{v:.1%}", ha="center", fontsize=7)
fig.delaxes(axes[len(bin_cols)])
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))

age_rate = df.groupby("Age")[TARGET].mean()
axes[0].plot(age_rate.index, age_rate.values, marker="o", color="#E45756")
axes[0].set_title("Tỉ lệ Diabetes theo nhóm tuổi (Age 1=18-24 → 13=80+)")
axes[0].set_xlabel("Age (mã)")
axes[0].set_ylabel("Tỉ lệ Diabetes")
axes[0].grid(True, alpha=0.3)

df["_bmi_bin"] = pd.cut(df["BMI"], bins=[0,18.5,25,30,35,100], labels=["Gầy","BT","Thừa cân","Béo I","Béo II+"])
bmi_rate = df.groupby("_bmi_bin", observed=False)[TARGET].mean()
axes[1].bar(bmi_rate.index.astype(str), bmi_rate.values, color="#4C78A8")
axes[1].set_title("Tỉ lệ Diabetes theo nhóm BMI")
axes[1].set_ylabel("Tỉ lệ Diabetes")
for j, v in enumerate(bmi_rate.values):
    axes[1].text(j, v+0.005, f"{v:.1%}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()
df = df.drop(columns=["_bmi_bin"])

## 4. Kiểm tra missing value nếu có


In [ ]:
TARGET_COL = TARGET
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

continuous_cols = ["BMI","MentHlth","PhysHlth"]  # liên tục / đếm ngày
ordinal_cols = ["GenHlth","Age","Education","Income"]  # thứ bậc 1..n
binary_cols = [c for c in FEATURE_COLS if c not in continuous_cols + ordinal_cols]

print("Continuous:", continuous_cols)
print("Ordinal   :", ordinal_cols)
print("Binary    :", binary_cols)

print("\nMissing tổng:", df[FEATURE_COLS].isna().sum().sum())
display(df[FEATURE_COLS].isna().sum().sort_values(ascending=False).head(10))

## 4. CHIA TẬP DỮ LIỆU TRAIN / VALIDATION / TEST (STRATIFIED SPLIT)
* Loại bỏ các bản ghi trùng lặp hoàn toàn trước khi chia để tránh việc cùng một bệnh nhân xuất hiện ở cả tập Train và tập Test (gây Data Leakage).
* Chia dữ liệu theo tỷ lệ **70% Train - 15% Validation - 15% Test** với tham số `stratify=y` để bảo toàn tỷ lệ nhãn mất cân bằng trên cả 3 tập.

In [ ]:
# 1. Loại bỏ các dòng trùng lặp hoàn toàn (Bộ BRFSS 2015 có khoảng ~24.000 bản ghi trùng)
initial_rows = len(df)
df_dedup = df.drop_duplicates().copy()
print(f"Đã loại bỏ {initial_rows - len(df_dedup):,} bản ghi trùng lặp. Số mẫu còn lại: {len(df_dedup):,}")

X = df_dedup.drop(columns=[TARGET])
y = df_dedup[TARGET].astype(int)

# 2. Chia Stratified Split: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print("\n── Kích thước và tỷ lệ nhãn Dương tính (1) trên từng tập ──")
for name, X_sub, y_sub in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    pos_rate = y_sub.mean() * 100
    print(f"{name:<12}: {X_sub.shape[0]:>7,} mẫu | {X_sub.shape[1]} đặc trưng | Tỷ lệ nhãn 1: {pos_rate:.2f}%")

## 5. LÀM SẠCH DỮ LIỆU & XỬ LÝ NGOẠI LAI (CHỈ FIT TRÊN TẬP TRAIN)
* Qua Boxplot ở phần EDA, biến `BMI` có nhiều giá trị ngoại lai cực đoan (lên tới 98.0), và các biến `MentHlth`, `PhysHlth` có phân phối lệch phải mạnh.
* **Nguyên tắc chống Data Leakage:** Ngưỡng cắt ngoại lai (Winsorization/Clipping theo phân vị 0.1% - 99% hoặc IQR) và giá trị điền khuyết (`SimpleImputer` với `median`) **chỉ được tính toán (`fit`) trên `X_train`**, sau đó áp ngưỡng đó để biến đổi (`transform`) cho `X_train`, `X_val`, và `X_test`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_clip=['BMI'], lower_quantile=0.001, upper_quantile=0.99):
        self.cols_to_clip = cols_to_clip
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        self.clip_bounds_ = {}
        self.imputer_ = SimpleImputer(strategy='median')
        self.feature_names_ = None

    def fit(self, X, y=None):
        X_df = X.copy()
        self.feature_names_ = X_df.columns.tolist()

        # 1. Học ngưỡng cắt ngoại lai (Outlier bounds) CHỈ TỪ TẬP TRAIN
        for col in self.cols_to_clip:
            if col in X_df.columns:
                q_low = X_df[col].quantile(self.lower_quantile)
                q_high = X_df[col].quantile(self.upper_quantile)
                self.clip_bounds_[col] = (q_low, q_high)

        # 2. Học giá trị trung vị (Median) để điền khuyết CHỈ TỪ TẬP TRAIN
        self.imputer_.fit(X_df)
        return self

    def transform(self, X):
        X_df = X.copy()

        # Điền khuyết (nếu có NaN phát sinh) bằng Median của tập Train
        X_imputed = pd.DataFrame(
            self.imputer_.transform(X_df),
            columns=self.feature_names_,
            index=X_df.index
        )

        # Cắt ngưỡng ngoại lai (Winsorization) bằng ngưỡng của tập Train
        for col, (q_low, q_high) in self.clip_bounds_.items():
            X_imputed[col] = X_imputed[col].clip(lower=q_low, upper=q_high)

        return X_imputed

# Khởi tạo bộ làm sạch: Chỉ fit trên X_train, sau đó transform cho cả 3 tập
cleaner = DataCleaner(cols_to_clip=['BMI'], lower_quantile=0.001, upper_quantile=0.99)
X_train_clean = cleaner.fit_transform(X_train)
X_val_clean   = cleaner.transform(X_val)
X_test_clean  = cleaner.transform(X_test)
print(f"Ngưỡng cắt ngoại lai BMI học từ tập Train: {cleaner.clip_bounds_['BMI']}")
print(f"Min - Max BMI trên tập Train sau khi làm sạch: {X_train_clean['BMI'].min():.1f} - {X_train_clean['BMI'].max():.1f}")

## 6. FEATURE ENGINEERING (ÁP DỤNG TRÊN ĐƠN VỊ ĐO GỐC)
Dùng hàm `add_engineered_features` đã định nghĩa ở phần EDA (các công thức cố định, không học tham số từ dữ liệu nên không gây leakage) và áp dụng lên Train / Validation / Test sau bước làm sạch:
1. `BMI_cat`: nhóm BMI (Gầy / Bình thường / Thừa cân / Béo I / Béo II+).
2. `BMI_x_GenHlth`: tương tác BMI × sức khỏe tổng quát.
3. `AgeGroup`: 4 nhóm tuổi (18-34, 35-54, 55-69, 70+).
4. `LifestyleScore` và `LifestyleLevel`: điểm / mức lối sống (hoạt động, rau, trái cây trừ hút thuốc, rượu).
5. `HealthBurden`: số điều kiện sức khỏe kém (sức khỏe tinh thần/thể chất >14 ngày, khó đi lại, đột quỵ, tim mạch).
6. `MetabolicRisk`: HighBP + HighChol + béo phì (BMI ≥ 30).

Hai cột `BMI_cat_name`, `AgeGroup_name` chỉ để vẽ biểu đồ nên được loại bỏ trước khi huấn luyện.

In [ ]:
def add_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    df2 = data.copy()

    df2["BMI_cat"] = pd.cut(
        df2["BMI"], bins=[0, 18.5, 25, 30, 35, 200], labels=False, include_lowest=True
    ).astype(float)
    bmi_label_map = {0:"Gầy", 1:"Bình thường", 2:"Thừa cân", 3:"Béo I", 4:"Béo II+"}
    df2["BMI_cat_name"] = df2["BMI_cat"].map(bmi_label_map)

    df2["BMI_x_GenHlth"] = df2["BMI"] * df2["GenHlth"]

    df2["AgeGroup"] = pd.cut(
        df2["Age"], bins=[0, 3, 7, 10, 13], labels=False, include_lowest=True
    ).astype(float)
    df2["AgeGroup_name"] = df2["AgeGroup"].map({0:"18-34", 1:"35-54", 2:"55-69", 3:"70+"})

    lifestyle = df2[["PhysActivity", "Fruits", "Veggies", "Smoker", "HvyAlcoholConsump"]].copy()
    lifestyle[["Smoker", "HvyAlcoholConsump"]] *= -1
    df2["LifestyleScore"] = lifestyle.sum(axis=1, min_count=5)
    df2["LifestyleLevel"] = pd.cut(
        df2["LifestyleScore"], bins=[-3, 0, 1, 4], labels=False, include_lowest=True
    ).astype(float)

    health_parts = pd.DataFrame({
        "mental": (df2["MentHlth"] > 14).where(df2["MentHlth"].notna()),
        "physical": (df2["PhysHlth"] > 14).where(df2["PhysHlth"].notna()),
        "walk": df2["DiffWalk"],
        "stroke": df2["Stroke"],
        "heart": df2["HeartDiseaseorAttack"],
    })
    df2["HealthBurden"] = health_parts.astype(float).sum(axis=1, min_count=5)

    obesity = (df2["BMI"] > 29.99).astype(float)
    obesity = obesity.where(df2["BMI"].notna())
    metabolic = pd.concat([df2[["HighBP", "HighChol"]], obesity.rename("obesity")], axis=1)
    df2["MetabolicRisk"] = metabolic.sum(axis=1, min_count=3)

    return df2

df_fe = add_engineered_features(df)
new_cols = ["BMI_cat", "BMI_cat_name", "BMI_x_GenHlth", "AgeGroup", "AgeGroup_name", "LifestyleScore", "LifestyleLevel", "HealthBurden", "MetabolicRisk"]
print(f"Đã tạo {len(new_cols)} cột mới. Shape: {df.shape} → {df_fe.shape}")
display(df_fe[new_cols + [TARGET]].head(8))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title in zip(
    axes,
    ["BMI_cat_name", "AgeGroup_name", "LifestyleLevel"],
    ["Diabetes theo nhóm BMI", "Diabetes theo AgeGroup", "Diabetes theo LifestyleLevel"],
):
    rate = df_fe.groupby(col, observed=False)[TARGET].mean()
    ax.bar(rate.index.astype(str), rate.values, color="#4C78A8")
    ax.set_title(title, fontsize=9)
    ax.set_ylabel("Tỉ lệ Diabetes")
    for j, v in enumerate(rate.values):
        ax.text(j, v + 0.005, f"{v:.1%}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

ENGINEERED_NUM = ["BMI_x_GenHlth", "LifestyleScore"]
ENGINEERED_CAT = ["BMI_cat", "AgeGroup", "LifestyleLevel", "HealthBurden", "MetabolicRisk"]
print("Engineered numeric:", ENGINEERED_NUM)
print("Engineered categorical:", ENGINEERED_CAT)
NAME_COLS = ["BMI_cat_name", "AgeGroup_name"]  # cột chữ chỉ dùng để trực quan hóa

def apply_fe(X_clean):
    return add_engineered_features(X_clean).drop(columns=NAME_COLS)

X_train_fe = apply_fe(X_train_clean)
X_val_fe   = apply_fe(X_val_clean)
X_test_fe  = apply_fe(X_test_clean)

print(f"Số lượng đặc trưng trước FE: {X_train_clean.shape[1]} -> Sau FE: {X_train_fe.shape[1]}")
display(X_train_fe.head(3))


## 7. CHUẨN HÓA THANG ĐO (SCALING — FIT TRÊN TRAIN, ĐẶT TRƯỚC SMOTE)
* Tách biệt **nhóm biến nhị phân (`0/1`)** (giữ nguyên) và **nhóm biến liên tục / thứ bậc / biến mới tạo** (chuẩn hóa bằng `StandardScaler`).


In [ ]:
# Tự phát hiện biến nhị phân (chỉ chứa 0/1) từ tập Train sau FE
binary_cols = [c for c in X_train_fe.columns if set(X_train_fe[c].unique()) <= {0.0, 1.0}]
scale_cols  = [c for c in X_train_fe.columns if c not in binary_cols]

print(f"Biến nhị phân giữ nguyên ({len(binary_cols)}): {binary_cols}")
print(f"Biến được chuẩn hóa ({len(scale_cols)}): {scale_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), scale_cols),
        ('bin_passthrough', 'passthrough', binary_cols)
    ],
    verbose_feature_names_out=False
)

# CHỈ FIT TRÊN X_TRAIN_FE, sau đó TRANSFORM cho cả 3 tập
all_cols_ordered = scale_cols + binary_cols
X_train_scaled = pd.DataFrame(preprocessor.fit_transform(X_train_fe), columns=all_cols_ordered, index=X_train_fe.index)
X_val_scaled   = pd.DataFrame(preprocessor.transform(X_val_fe),   columns=all_cols_ordered, index=X_val_fe.index)
X_test_scaled  = pd.DataFrame(preprocessor.transform(X_test_fe),  columns=all_cols_ordered, index=X_test_fe.index)

display(X_train_scaled[scale_cols].describe().T[['mean', 'std', 'min', 'max']].round(2))


## 8. CHỌN LỌC ĐẶC TRƯNG (FEATURE SELECTION — TRÊN TẬP TRAIN TRƯỚC SMOTE)
* Sử dụng phương pháp Embedded (`SelectFromModel` kết hợp `RandomForestClassifier` có `class_weight='balanced'`) chỉ chạy trên `(X_train_scaled, y_train)`.


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Dùng Random Forest trên tập Train gốc (chưa SMOTE) để đánh giá độ quan trọng đặc trưng thực tế
rf_selector_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# Giữ lại các đặc trưng có độ quan trọng >= 0.5 × mức trung bình
selector = SelectFromModel(estimator=rf_selector_model, threshold='0.5*mean')
selector.fit(X_train_scaled, y_train)

selected_features = X_train_scaled.columns[selector.get_support()].tolist()
dropped_features  = [c for c in X_train_scaled.columns if c not in selected_features]

print(f"Tổng số đặc trưng được giữ lại: {len(selected_features)}/{X_train_scaled.shape[1]}")
print("── Các đặc trưng được chọn ──\n", selected_features)
print("── Các đặc trưng bị loại bỏ ──\n", dropped_features)

# Lọc cột đồng nhất cho cả Train, Val và Test
X_train_sel = X_train_scaled[selected_features].copy()
X_val_sel   = X_val_scaled[selected_features].copy()
X_test_sel  = X_test_scaled[selected_features].copy()

# Vẽ biểu đồ Feature Importance học từ tập Train
importances = pd.Series(selector.estimator_.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=True)
plt.figure(figsize=(9, 7))
colors = ['#4C78A8' if col in selected_features else '#D3D3D3' for col in importances.index]
importances.plot(kind='barh', color=colors)
plt.axvline(selector.threshold_, color='#E45756', linestyle='--', label=f'Ngưỡng lọc ({selector.threshold_:.4f})')
plt.title("Độ quan trọng đặc trưng (Feature Importance) trên tập Train", weight='bold')
plt.xlabel("Gini Importance")
plt.legend()
plt.tight_layout()
plt.show()

## 9. CÂN BẰNG DỮ LIỆU VỚI SMOTE (CHỈ ÁP DỤNG TRÊN TẬP TRAIN)
* **Nguyên tắc sống còn:** Chỉ dùng `SMOTE` để sinh mẫu nhân tạo trên `(X_train_sel, y_train)`. Hai tập `Validation` và `Test` giữ nguyên phân phối mất cân bằng tự nhiên.
* Thiết lập `sampling_strategy=1`

In [ ]:
print("── Trước khi SMOTE (Tập Train) ──")
print(Counter(y_train))

# Áp dụng SMOTE chỉ trên tập Train đã qua chọn lọc đặc trưng
smote = SMOTE(sampling_strategy=1, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_sel, y_train)

print("\n── Sau khi SMOTE (Tập Train) ──")
print(Counter(y_train_smote))
print(f"Tỷ lệ Dương tính mới trên tập Train: {y_train_smote.mean()*100:.2f}%")
print(f"Tập Validation giữ nguyên: {Counter(y_val)} ({y_val.mean()*100:.2f}% Dương tính)")
print(f"Tập Test giữ nguyên      : {Counter(y_test)} ({y_test.mean()*100:.2f}% Dương tính)")

## 10. HUẤN LUYỆN THUẬT TOÁN, TỐI ƯU NGƯỠNG (THRESHOLD TUNING) & ĐÁNH GIÁ

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score,
    recall_score, confusion_matrix
)

# KHÔNG bật class_weight='balanced' vì dữ liệu Train đã được SMOTE
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=RANDOM_STATE),
}


val_results = []
val_probas = {}

for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)

    # Predict probabilities on the original Validation set
    y_val_prob = model.predict_proba(X_val_sel)[:, 1]

    # Predict labels using the default threshold of 0.5
    y_val_pred_default = (y_val_prob >= 0.5).astype(int)

    # Store probabilities for threshold tuning later
    val_probas[name] = y_val_prob

    val_results.append({
        "Model": name,
        "Recall (τ=0.5)": recall_score(
            y_val, y_val_pred_default, zero_division=0
        ),
        "Precision (τ=0.5)": precision_score(
            y_val, y_val_pred_default, zero_division=0
        ),
        "F1-Score (τ=0.5)": f1_score(
            y_val, y_val_pred_default, zero_division=0
        ),
    })

results_df = pd.DataFrame(val_results)

print("── KẾT QUẢ SO SÁNH CÁC THUẬT TOÁN TRÊN TẬP VALIDATION (NGƯỠNG MẶC ĐỊNH 0.5) ──")
display(results_df.round(4))


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ══════════════════════════════════════════════════════════════
# MỤC TIÊU LÂM SÀNG: Recall >= TARGET_RECALL, sau đó Precision cao nhất
# (cả việc CHỌN MÔ HÌNH lẫn CHỌN NGƯỠNG đều chỉ dùng tập Validation)
# ══════════════════════════════════════════════════════════════
TARGET_RECALL = 0.80
thresholds = np.arange(0.05, 0.95, 0.01)

def sweep_thresholds(y_true, prob):
    """Tính Recall / Precision của từng ngưỡng trong `thresholds`."""
    preds = [(prob >= t).astype(int) for t in thresholds]
    rec  = np.array([recall_score(y_true, p, zero_division=0) for p in preds])
    prec = np.array([precision_score(y_true, p, zero_division=0) for p in preds])
    return rec, prec

def pick_threshold(rec, prec, target=TARGET_RECALL):
    """Trong các ngưỡng có Recall >= target, chọn ngưỡng có Precision cao nhất.
    Trả về (chỉ số ngưỡng, có đạt mục tiêu hay không)."""
    feasible = rec >= target
    if feasible.any():
        return int(np.flatnonzero(feasible)[np.argmax(prec[feasible])]), True
    return int(np.argmax(rec)), False   # không ngưỡng nào đạt -> lấy ngưỡng có Recall cao nhất

# ── 1) So sánh các mô hình theo cùng một mục tiêu ──
sweeps, rows = {}, []
for name, prob in val_probas.items():
    rec, prec = sweep_thresholds(y_val, prob)
    idx, reached = pick_threshold(rec, prec)
    sweeps[name] = (rec, prec, idx, reached)
    rows.append({
        "Model": name,
        "Đạt Recall mục tiêu": reached,
        "τ": round(float(thresholds[idx]), 2),
        "Recall (Val)": rec[idx],
        "Precision (Val)": prec[idx],
    })

compare_df = pd.DataFrame(rows)
print(f"── SO SÁNH MÔ HÌNH TẠI MỤC TIÊU Recall ≥ {TARGET_RECALL:.2f} (Validation) ──")
display(compare_df.round(4))

# Ưu tiên mô hình đạt mục tiêu; trong đó chọn Precision cao nhất
# (nếu không mô hình nào đạt, lấy mô hình có Recall cao nhất)
if compare_df["Đạt Recall mục tiêu"].any():
    cand = compare_df[compare_df["Đạt Recall mục tiêu"]]
    best_model_name = cand.loc[cand["Precision (Val)"].idxmax(), "Model"]
else:
    print(f"⚠ Không mô hình nào đạt Recall >= {TARGET_RECALL:.2f} trên Validation; chọn mô hình có Recall cao nhất.")
    best_model_name = compare_df.loc[compare_df["Recall (Val)"].idxmax(), "Model"]

best_model = models[best_model_name]
val_recall, val_precision, idx, _ = sweeps[best_model_name]
best_t = float(thresholds[idx])

print(f"\nMô hình được chọn: {best_model_name}")
print(f"  + Ngưỡng τ* = {best_t:.2f} "
      f"(Validation: Recall = {val_recall[idx]:.4f}, Precision = {val_precision[idx]:.4f})")

# ── 2) ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST (CHỈ CHẠY 1 LẦN) ──
y_test_prob = best_model.predict_proba(X_test_sel)[:, 1]
y_test_pred = (y_test_prob >= best_t).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_auc = roc_auc_score(y_test, y_test_prob)

print(f"\n===== KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST ({best_model_name}, τ* = {best_t:.2f}) =====")
print(f"Test ROC-AUC    : {test_auc:.4f}")
print(f"Test Recall     : {recall_score(y_test, y_test_pred):.4f}  (bỏ sót {fn:,}/{fn + tp:,} ca bệnh)")
print(f"Test Precision  : {precision_score(y_test, y_test_pred):.4f}")
print(f"Test F1         : {f1_score(y_test, y_test_pred):.4f}\n")
print(classification_report(y_test, y_test_pred, target_names=["Không tiểu đường (0)", "Tiểu đường/Tiền TĐ (1)"]))

# ── 3) Biểu đồ ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# (a) Recall / Precision theo ngưỡng trên Validation
ax = axes[0]
ax.plot(thresholds, val_recall, label="Recall (Validation)", color="#E45756", lw=2)
ax.plot(thresholds, val_precision, label="Precision (Validation)", color="#4C78A8", lw=2)
ax.axhline(TARGET_RECALL, color="gray", linestyle=":", label=f"Mục tiêu Recall = {TARGET_RECALL:.2f}")
ax.axvline(best_t, color="black", linestyle="--", label=f"Ngưỡng chọn τ* = {best_t:.2f}")
ax.set_title(f"Chọn ngưỡng theo ràng buộc Recall (Validation) — {best_model_name}")
ax.set_xlabel("Ngưỡng xác suất (τ)")
ax.set_ylabel("Giá trị")
ax.set_ylim(0, 1.02)
ax.legend(loc="center left", fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)

# (b) Đường ROC trên Test + điểm vận hành tại τ*
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
ax.plot(fpr, tpr, color="#4C78A8", lw=2, label=f"{best_model_name} (AUC = {test_auc:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Ngẫu nhiên (AUC = 0.5)")
ax.scatter([fp / (fp + tn)], [tp / (tp + fn)], color="#E45756", s=70, zorder=5,
           label=f"Điểm vận hành τ* = {best_t:.2f}\n(TPR = {tp / (tp + fn):.2f}, FPR = {fp / (fp + tn):.2f})")
ax.set_title("Đường ROC trên tập Test")
ax.set_xlabel("False Positive Rate (1 − Specificity)")
ax.set_ylabel("True Positive Rate (Recall)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 11. XUẤT MÔ HÌNH ĐỂ CHẠY STREAMLIT

Chạy ô code tiếp theo sau khi toàn bộ các ô huấn luyện và đánh giá đã chạy thành công. Ô này lưu Logistic Regression, ngưỡng và các bước tiền xử lý cần thiết cho ứng dụng.

In [ ]:
# DÁN TOÀN BỘ Ô NÀY VÀO MỘT CELL MỚI Ở CUỐI NOTEBOOK SAU KHI CELL 27 CHẠY THÀNH CÔNG.
# Cell này xuất Logistic Regression cùng toàn bộ bước tiền xử lý mà app.py cần.

from pathlib import Path
import joblib
import sklearn
import pandas as pd
import numpy as np

# Lấy riêng mô hình Logistic Regression đã huấn luyện trong notebook.
# Ngưỡng được chọn theo quy tắc Recall >= 0.80 trên tập Validation.
lr_model = models["Logistic Regression"]
lr_recall, lr_precision, lr_idx, lr_reached_target = sweeps["Logistic Regression"]
lr_threshold = float(thresholds[lr_idx])

model_bundle = {
    "model": lr_model,
    "threshold": lr_threshold,
    "cleaner": cleaner,
    "preprocessor": preprocessor,
    "selected_features": list(selected_features),
    "scale_cols": list(scale_cols),
    "binary_cols": list(binary_cols),
    "feature_cols": list(X.columns),
    "validation_recall_at_threshold": float(lr_recall[lr_idx]),
    "validation_precision_at_threshold": float(lr_precision[lr_idx]),
    "validation_recall_target_reached": bool(lr_reached_target),
    "library_versions": {
        "python_note": "Use a Python environment compatible with the installed scikit-learn version.",
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "joblib": joblib.__version__,
    },
}

output_path = Path("diabetes_model_bundle.joblib")
joblib.dump(model_bundle, output_path)

print(f"Đã lưu mô hình: {output_path.resolve()}")
print(f"Ngưỡng phân loại Logistic Regression: {lr_threshold:.2f}")
print(f"Recall Validation tại ngưỡng này: {lr_recall[lr_idx]:.4f}")
print(f"Precision Validation tại ngưỡng này: {lr_precision[lr_idx]:.4f}")
print("Phiên bản thư viện:", model_bundle["library_versions"])

# Nếu chạy trên Google Colab, mở hộp thoại tải tệp về máy.
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print("Tệp đã nằm trong thư mục làm việc hiện tại. Hãy chép nó vào cùng thư mục với app.py.")
